# 🐍 Agentic Protocols with Microsoft Agent Framework (Python)

## 📋 Scenario Overview

This notebook demonstrates **agentic communication protocols** using the Microsoft Agent Framework. We'll explore how agents can interact with external services and other agents through standardized protocols.

**Key Concepts:**
- 🔌 **Protocol Integration**: Connect agents to external services
- 🤝 **Agent Interoperability**: Enable agents to work with different systems
- 🌐 **Service Discovery**: Dynamically discover available capabilities
- 📡 **Standardized Communication**: Use industry-standard protocols

## 🏗️ Technical Implementation

### Core Components
- **Agent Framework**: Python implementation of Microsoft's agent orchestration
- **Azure OpenAI**: GPT-4o-mini for natural language understanding
- **Microsoft Entra ID**: Secure keyless authentication
- **External Services**: Integration with third-party APIs

### Protocol Flow
```
User Request → Agent → Protocol Interface → External Service
                                ↓
                          Response
                                ↓
                    Process & Format
                                ↓
                         User Response
```

## ⚙️ Prerequisites

**Required:**
```bash
pip install agent-framework-core azure-identity python-dotenv httpx
```

**Environment Variables (.env):**
```env
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-10-21
```

**Azure RBAC:** "Cognitive Services OpenAI User" role required

Let's build a protocol-aware agent! 🌟

In [ ]:
# Package check - Install manually if needed
try:
    import agent_framework
    import httpx
    print("✓ agent-framework-core is installed")
    print("✓ httpx is installed")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Please install: pip install agent-framework-core httpx -U")
    raise

In [ ]:
# 📦 Import Required Libraries
import os
import json
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import InteractiveBrowserCredential
from typing import Annotated
from IPython.display import display, HTML
import httpx

In [ ]:
# 🔧 Load Environment Variables
load_dotenv(override=True)

# Verify configuration
print(f"✓ AZURE_OPENAI_ENDPOINT: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"✓ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
print(f"✓ AZURE_OPENAI_API_VERSION: {os.environ.get('AZURE_OPENAI_API_VERSION')}")

# Critical: Remove API key if present to force Entra ID authentication
if os.environ.get('AZURE_OPENAI_API_KEY'):
    del os.environ['AZURE_OPENAI_API_KEY']
    print("✓ API key removed - using Entra ID authentication")

In [ ]:
# 🌐 External Service Integration: Currency Exchange API
# This demonstrates how to integrate external HTTP APIs with Agent Framework

def get_exchange_rate(currency_from: str, currency_to: str, date: str = "latest") -> str:
    """Retrieves exchange rate between currencies using Frankfurter API.
    
    Args:
        currency_from: Currency code to convert from (e.g., USD)
        currency_to: Currency code to convert to (e.g., EUR, INR)
        date: Date for historical rates or 'latest' for current
    
    Returns:
        str: Exchange rate information or error message
    """
    try:
        response = httpx.get(
            f'https://api.frankfurter.app/{date}',
            params={'from': currency_from, 'to': currency_to},
            timeout=10.0,
        )
        response.raise_for_status()
        data = response.json()
        
        if 'rates' not in data or currency_to not in data['rates']:
            return f'Could not retrieve rate for {currency_from} to {currency_to}'
        
        rate = data['rates'][currency_to]
        return f'1 {currency_from} = {rate} {currency_to}'
    except Exception as e:
        return f'Currency API call failed: {str(e)}'

def get_travel_destinations() -> str:
    """Provides a list of popular travel destinations.
    
    Returns:
        str: List of destinations with brief descriptions
    """
    return """
    Popular Travel Destinations:
    1. Barcelona, Spain - Mediterranean beaches, Gaudí architecture
    2. Paris, France - Art, culture, and romantic atmosphere
    3. Tokyo, Japan - Modern technology meets ancient traditions
    4. New York, USA - The city that never sleeps
    5. Dubai, UAE - Luxury shopping and modern architecture
    """

def get_destination_info(destination: str) -> str:
    """Provides detailed information about a travel destination.
    
    Args:
        destination: Name of the destination
    
    Returns:
        str: Detailed information about the destination
    """
    destinations = {
        "Barcelona": {
            "currency": "EUR",
            "language": "Spanish, Catalan",
            "best_time": "May-June, September-October",
            "highlights": "Sagrada Familia, Park Güell, Las Ramblas"
        },
        "Paris": {
            "currency": "EUR",
            "language": "French",
            "best_time": "April-June, October-November",
            "highlights": "Eiffel Tower, Louvre, Notre-Dame"
        },
        "Tokyo": {
            "currency": "JPY",
            "language": "Japanese",
            "best_time": "March-May, September-November",
            "highlights": "Senso-ji Temple, Tokyo Tower, Shibuya Crossing"
        },
        "New York": {
            "currency": "USD",
            "language": "English",
            "best_time": "April-June, September-November",
            "highlights": "Statue of Liberty, Central Park, Times Square"
        },
        "Dubai": {
            "currency": "AED",
            "language": "Arabic, English",
            "best_time": "November-March",
            "highlights": "Burj Khalifa, Dubai Mall, Palm Jumeirah"
        }
    }
    
    city = destination.split(',')[0].strip()
    
    if city in destinations:
        info = destinations[city]
        return f"""
        {city} Information:
        - Currency: {info['currency']}
        - Language: {info['language']}
        - Best Time to Visit: {info['best_time']}
        - Top Highlights: {info['highlights']}
        """
    else:
        return f"Information not available for {destination}"

print("✓ External service integration functions defined")

In [ ]:
# 🔗 Create Azure OpenAI Chat Client with Browser Authentication

print("🔐 Authenticating with Azure...")
print("   A browser window will open for you to sign in")

# Create credential that will open browser for authentication
credential = InteractiveBrowserCredential()

# Create Azure OpenAI client
openai_chat_client = AzureOpenAIChatClient(
    credential=credential,
    endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION")
)

print(f"✓ Connected to Azure OpenAI")
print(f"  Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")

In [ ]:
# 🤖 Create the Travel Planning Agent with Protocol Integration

AGENT_INSTRUCTIONS = """
You are a comprehensive travel planning assistant with access to real-time data through external services.

You have access to the following capabilities:
1. get_travel_destinations: Get a list of popular travel destinations
2. get_destination_info: Get detailed information about a specific destination
3. get_exchange_rate: Get current or historical currency exchange rates

Your process for assisting users:
- When users ask about destinations, provide comprehensive information including currency, language, and highlights
- Always offer to check exchange rates when discussing international destinations
- Provide practical travel advice based on the destination information
- Use the external currency API to provide real-time exchange rates
- Format responses clearly with proper sections and bullet points

Guidelines:
- Be enthusiastic and helpful about travel planning
- Provide accurate, up-to-date information using the available tools
- Suggest checking exchange rates when relevant
- Offer to provide more details about any destination
- Be transparent about which information comes from external services

Your goal is to be a knowledgeable travel assistant that leverages external services to provide the best possible information.
"""

agent = ChatAgent(
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    tools=[get_travel_destinations, get_destination_info, get_exchange_rate]
)

print("✓ Travel planning agent created with external service integration")

In [ ]:
# 💬 Example 1: Destination Information Request

user_inputs = [
    "Tell me about Barcelona and what's the current USD to EUR exchange rate?",
]

print("🎯 Example 1: Destination info with currency exchange...\n")

async def run_conversation():
    for user_input in user_inputs:
        # Display user message
        html_output = (
            f"<div style='margin-bottom:10px'>" 
            f"<div style='font-weight:bold; color:#0066cc;'>👤 User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )
        
        # Get agent response
        response = await agent.run(user_input)
        
        # Display agent response
        html_output += (
            f"<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold; color:#009900;'>🤖 TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{response}</div></div><hr>"
        )
        
        display(HTML(html_output))

await run_conversation()

print("\n✅ External API integration demonstrated!")

In [ ]:
# 💬 Example 2: Multi-Destination Comparison

user_inputs = [
    "Compare Tokyo and Paris for a trip, and show me USD exchange rates for both",
]

print("🎯 Example 2: Multi-destination comparison with currencies...\n")

async def run_comparison():
    for user_input in user_inputs:
        # Display user message
        html_output = (
            f"<div style='margin-bottom:10px'>" 
            f"<div style='font-weight:bold; color:#0066cc;'>👤 User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )
        
        # Get agent response
        response = await agent.run(user_input)
        
        # Display agent response
        html_output += (
            f"<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold; color:#009900;'>🤖 TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{response}</div></div><hr>"
        )
        
        display(HTML(html_output))

await run_comparison()

print("\n✅ Multi-service orchestration demonstrated!")

## 🎓 Key Takeaways: Agentic Protocols

**What We Demonstrated:**
1. ✅ **External API Integration**: Connected to Frankfurter currency API
2. ✅ **Service Composition**: Combined multiple data sources
3. ✅ **Protocol Abstraction**: Agent doesn't need to know HTTP details
4. ✅ **Error Handling**: Graceful handling of API failures

**Protocol Patterns:**
- **REST APIs**: HTTP-based services like currency exchange
- **Function Wrapping**: Python functions hide protocol complexity
- **Async Operations**: Non-blocking calls to external services
- **Response Formatting**: Structured data transformed for LLM consumption

**Real-World Applications:**
- 🌐 **API Aggregation**: Combine data from multiple sources
- 🤝 **Agent-to-Agent (A2A)**: Agents communicating with other agents
- 📡 **Model Context Protocol (MCP)**: Standardized tool integration
- 🔌 **Plugin Architecture**: Extensible agent capabilities

**Comparison with Semantic Kernel:**
- **Agent Framework**: Tools are simple Python functions
- **Semantic Kernel**: Uses `@kernel_function` decorators and plugins
- **Both**: Support async operations and error handling
- **Agent Framework**: More Pythonic, less ceremony

**Advanced Protocols:**
- **A2A (Agent-to-Agent)**: Multi-agent orchestration (see Semantic Kernel A2A example)
- **MCP (Model Context Protocol)**: Standardized tool discovery and invocation
- **WebSocket**: Real-time bidirectional communication
- **gRPC**: High-performance RPC for distributed agents

**Best Practices:**
- Implement timeout handling for external APIs
- Cache responses when appropriate
- Validate external data before using it
- Log all external service calls
- Implement rate limiting

**Next Steps:**
- Explore A2A protocol with Semantic Kernel (Lesson 11 A2A notebook)
- Implement MCP servers for custom tools
- Build multi-agent systems with specialized agents
- Add monitoring and observability

---

**🎉 Congratulations!** You've built an agent that integrates with external services using protocol patterns!